## 1. What are Tools?

**Tools** are external functions or capabilities that an AI model can use to perform tasks that it cannot reliably do by itself.

Examples:

- Search the web
- Perform calculations
- Query a database
- Call an API
- Read or write files
- Get weather information

## 2. Tool Calling

**Tool calling** allows a model to decide when it needs to use a particular tool and provide the required arguments to that tool.

The model does **not directly execute the tool**. Instead, it generates a **tool call**, and the application executes the tool.

### Basic Flow

```text
User
  ↓
LLM
  ↓
Decides a tool is needed
  ↓
Tool Call + Arguments
  ↓
Application executes Tool
  ↓
Tool Result
  ↓
LLM
  ↓
Final Response

In [1]:
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3.8-27b")
response = model.invoke("How are you?", max_tokens=256)
print(response.content)

I'm doing well, thank you for asking! I'm ready and eager to help you with whatever you need.

How are you doing today? Is there something specific on your mind or a task I can assist you with?


In [4]:
# Tool calling
from langchain.tools import tool
# @tool is a decorator that allows you to define a function as a tool that can be called by the model.
@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    # In a real implementation, you would call a weather API here
    return f"The current weather in {location} is sunny with a temperature of 25°C."


model_with_tool = model.bind_tools([get_current_weather])

In [7]:
response_with_tool = model_with_tool.invoke("What is the current weather in New York?", max_tokens=256)
print(response_with_tool)
for tool_call in response_with_tool.tool_calls:
    print(f"Tool called: {tool_call['name']} with arguments: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'jshs03e3f', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 282, 'total_tokens': 310, 'completion_time': 0.074626343, 'completion_tokens_details': None, 'prompt_time': 0.019730775, 'prompt_tokens_details': None, 'queue_time': 0.00874831, 'total_time': 0.094357118}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0fcf0-3746-7023-ac59-3cbd2ec8abfd-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'jshs03e3f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 282, 'output_tokens': 28, 'total_tokens': 310}
Tool called: get_current_weather with arguments: {'location': 'New York'}


<br><br>

## Tool Execution Loop

**The Tool Execution Loop is simply a cycle where the model asks for a tool → the tool runs → the result goes back to the model → the model gives the answer.**



In [11]:
# model generate tool call

messages = [
    {"role": "user", "content": "What is the weather of Paris?"}
]
ai_msg = model_with_tool.invoke(messages, max_tokens=256)
messages.append(ai_msg)


# execute tool cll and get result

for tool_call in ai_msg.tool_calls:
    tool_result = get_current_weather.invoke(tool_call)
    messages.append(tool_result)

# pass result back to model

final_response = model_with_tool.invoke(messages, max_tokens=256)
print(final_response.text)



The current weather in Paris is sunny with a temperature of 25°C.
